In [1]:
from datasets import load_dataset
import numpy as np
import evaluate
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaModel, RobertaForSequenceClassification, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import get_peft_model, LoraConfig, TaskType, PeftModel, PeftConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "roberta-base"

c:\Users\lewy7\Documents\GitHub\roberta-hypernet-custom\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = RobertaTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=512)

dataset = load_dataset("glue", "cola")

encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

metric = evaluate.load("glue", "cola")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [3]:
# --- Hypernetwork: A -> B ---
class LoRAHyperNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, lora_dim):
        super().__init__()
        self.fc1 = nn.Linear(lora_dim * input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, input_dim * lora_dim)

    def forward(self, A):
        flat = A.view(-1)
        h = torch.relu(self.fc1(flat))
        B = self.fc2(h).view(A.size(1), A.size(0))  # B shape: [in_dim, r]
        return B

In [21]:
class RobertaWithDynamicLoRA(RobertaForSequenceClassification):
    def __init__(self, config, lora_r=8, hypernet_hidden_dim=128):
        super().__init__(config)
        self.config.output_hidden_states = True 
        self.hidden_size = config.hidden_size
        self.lora_r = lora_r

        # Create hypernetwork
        self.hypernet = LoRAHyperNet(self.hidden_size, hypernet_hidden_dim, lora_r)

        # Use fixed A in eval, learned through computation in training
        self.register_buffer("A_fixed", torch.randn(lora_r, self.hidden_size))

        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        device = input_ids.device
        
        print(self.hypernet.fc1.weight[:5][:5]) # check if the gradients flow properly throught the hypernetwork
        
        A = torch.randn((self.lora_r, self.hidden_size), device=device) # [r, hidden_size]
        B = self.hypernet(A)  # [hidden_size, r]

        # Forward pass through model to get hidden states
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True
        )
        hidden_states = outputs.hidden_states[-1]  # [batch, seq_len, hidden_size]

        # Apply dynamic LoRA on final layer hidden states
        # hidden + (hidden @ A.T @ B.T)
        lora_out = torch.matmul(hidden_states, A.T)     # [batch, seq_len, r]
        lora_out = torch.matmul(lora_out, B.T)          # [batch, seq_len, hidden_size]
        adapted_hidden = hidden_states + lora_out

        # Use [CLS] token from adapted hidden
        cls_output = adapted_hidden[:, 0]  # [batch, hidden]
        logits = self.dropout(cls_output)
        logits = self.classifier.out_proj(logits)

        loss = None
        if labels is not None:
            loss_fn = torch.nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"logits": logits, "loss": loss} if loss is not None else {"logits": logits}


In [22]:
base_model = RobertaWithDynamicLoRA.from_pretrained(model_name)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=base_model.lora_r,
    lora_alpha=8,
    lora_dropout=0.1
)

base_model = get_peft_model(base_model, peft_config=peft_config)

for param in base_model.hypernet.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Some weights of RobertaWithDynamicLoRA were not initialized from the model checkpoint at roberta-base and are newly initialized: ['A_fixed', 'classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight', 'hypernet.fc1.bias', 'hypernet.fc1.weight', 'hypernet.fc2.bias', 'hypernet.fc2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 127,113,348
Trainable parameters: 2,466,178


In [23]:
for name, param in base_model.named_parameters():
    if "hypernet" in name:
        print(name, param.requires_grad)

base_model.model.hypernet.fc1.weight True
base_model.model.hypernet.fc1.bias True
base_model.model.hypernet.fc2.weight True
base_model.model.hypernet.fc2.bias True


In [24]:
print(base_model.roberta.encoder.layer[-1].attention.self.query.lora_B.default.weight.shape)

torch.Size([768, 8])


In [25]:
training_args = TrainingArguments(
    output_dir="./outputs/reoberta_base_cola",
    eval_strategy="epoch",
    # eval_steps=25,
    save_strategy="steps",
    save_steps=1000000000,
    learning_rate=4e-4,
    per_device_train_batch_size=16, # 16
    gradient_accumulation_steps=2, # 2
    per_device_eval_batch_size=32,
    num_train_epochs=1, # 80
    logging_dir="./logs/roberta_base_cola",
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    disable_tqdm=False,
    # remove_unused_columns=False
)

In [26]:
trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [27]:
trainer.train()

tensor([[-0.0219, -0.0617, -0.0071,  ...,  0.0079,  0.0266,  0.0295],
        [-0.0107, -0.0282, -0.0157,  ..., -0.0265, -0.0143,  0.0054],
        [-0.0193, -0.0087, -0.0006,  ...,  0.0071, -0.0264, -0.0016],
        [-0.0257, -0.0354,  0.0295,  ..., -0.0277, -0.0234, -0.0010],
        [-0.0104,  0.0214,  0.0185,  ..., -0.0003,  0.0255, -0.0237]],
       device='cuda:0', grad_fn=<SliceBackward0>)
tensor([[-0.0219, -0.0617, -0.0071,  ...,  0.0079,  0.0266,  0.0295],
        [-0.0107, -0.0282, -0.0157,  ..., -0.0265, -0.0143,  0.0054],
        [-0.0193, -0.0087, -0.0006,  ...,  0.0071, -0.0264, -0.0016],
        [-0.0257, -0.0354,  0.0295,  ..., -0.0277, -0.0234, -0.0010],
        [-0.0104,  0.0214,  0.0185,  ..., -0.0003,  0.0255, -0.0237]],
       device='cuda:0', grad_fn=<SliceBackward0>)


Epoch,Training Loss,Validation Loss,Matthews Correlation
1,No log,2.634069,-0.019648


tensor([[-0.0219, -0.0617, -0.0071,  ...,  0.0079,  0.0266,  0.0295],
        [-0.0107, -0.0282, -0.0157,  ..., -0.0265, -0.0143,  0.0054],
        [-0.0193, -0.0087, -0.0006,  ...,  0.0071, -0.0264, -0.0016],
        [-0.0257, -0.0354,  0.0295,  ..., -0.0277, -0.0234, -0.0010],
        [-0.0104,  0.0214,  0.0185,  ..., -0.0003,  0.0255, -0.0237]],
       device='cuda:0', grad_fn=<SliceBackward0>)
tensor([[-0.0219, -0.0617, -0.0071,  ...,  0.0079,  0.0266,  0.0295],
        [-0.0107, -0.0282, -0.0157,  ..., -0.0265, -0.0143,  0.0054],
        [-0.0193, -0.0087, -0.0006,  ...,  0.0071, -0.0264, -0.0016],
        [-0.0257, -0.0354,  0.0295,  ..., -0.0277, -0.0234, -0.0010],
        [-0.0104,  0.0214,  0.0185,  ..., -0.0003,  0.0255, -0.0237]],
       device='cuda:0', grad_fn=<SliceBackward0>)
tensor([[-0.0219, -0.0618, -0.0071,  ...,  0.0079,  0.0267,  0.0296],
        [-0.0107, -0.0282, -0.0158,  ..., -0.0265, -0.0143,  0.0054],
        [-0.0192, -0.0087, -0.0007,  ...,  0.0071, -0.0265

TrainOutput(global_step=268, training_loss=8.113701208313898, metrics={'train_runtime': 403.5642, 'train_samples_per_second': 21.189, 'train_steps_per_second': 0.664, 'total_flos': 2314645855358976.0, 'train_loss': 8.113701208313898, 'epoch': 1.0})